<a href="https://colab.research.google.com/github/Mariam-Elbishbeashy/HeadlineGeneration-NLP/blob/main/headlineGenerationNLP_trainReadyPreprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Preprocessing Notebook — News Headline Generation

## Final goal

Create model-ready files:

- `model_input`: cleaned article text, optionally with a task prefix such as `summarize:`
- `model_target`: cleaned headline/title
- train / validation / test CSV files



## 1. Import Libraries


- `GroupShuffleSplit` to reduce leakage across train/val/test when articles are very similar
- optional `langdetect` for language filtering
- optional Hugging Face tokenizer checks for token-length filtering

In [33]:
import os
import re
import html
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

pd.set_option("display.max_colwidth", 160)

## Install Extra Packages in Colab

- `contractions` (for contraction expansion)
- `langdetect` (for language filtering)
- `transformers` (for tokenizer-based length checks)

Only True if this is the notebook first run
False if already installed after first run

In [34]:
INSTALL_OPTIONAL_PACKAGES = False

if INSTALL_OPTIONAL_PACKAGES:
    !pip -q install contractions langdetect transformers sentence-transformers
    print("Optional packages installed.")
else:
    print("Skipped package installation. Set INSTALL_OPTIONAL_PACKAGES=True if needed.")

Skipped package installation. Set INSTALL_OPTIONAL_PACKAGES=True if needed.


## 3. Mount Google Drive and Set Paths

Expected input columns:

- `article`
- `title`

Optional metadata columns:

- `date`
- `publication`
- `section`
- `author`
- `url`



In [35]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print("Not running inside Google Colab, or Drive mount skipped.")

# Change this path to your actual sampled dataset file.
INPUT_PATH = "/content/drive/MyDrive/headlineGenerationProjectNLP/sampled_articles_dataset.pkl"

OUTPUT_DIR = "/content/drive/MyDrive/headlineGenerationProjectNLP/news_headline_model_data"
if not os.path.exists("/content/drive/MyDrive"):
    OUTPUT_DIR = "/mnt/data/news_headline_model_data"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Output directory:", OUTPUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Output directory: /content/drive/MyDrive/headlineGenerationProjectNLP/news_headline_model_data


## 4. Load the Dataset

The sampled dataset already prepared, around 56k rows

At this stage, we are not doing linguistic analysis. We are only loading the base data that will eventually go to the model.

In [36]:
def load_dataframe(path):
    path = str(path)
    if path.endswith(".csv"):
        return pd.read_csv(path)
    if path.endswith(".pkl") or path.endswith(".pickle"):
        return pd.read_pickle(path)
    if path.endswith(".parquet"):
        return pd.read_parquet(path)
    raise ValueError("Unsupported file type. Use .csv, .pkl, .pickle, or .parquet")

raw_df = load_dataframe(INPUT_PATH)
print("Loaded shape:", raw_df.shape)
display(raw_df.head(3))
print("Columns:")
print(raw_df.columns.tolist())

Loaded shape: (55287, 14)


,idx,article_idx,date,year,month,day,author,title,article,url,section,publication,section_clean,meta_section
0,-8589,-8589,2018-03-18,2018,3.0,18,None,China's Didi looks to raise $1.6 billion via asset-backed securities,SHANGHAI (Reuters) - A unit of Chinese ride-hailing firm Didi Chuxing has submitted an application to raise 10 billion yuan ($1.6 billion) through an issuan...,https://www.reuters.com/article/didi-fundraising/chinas-didi-looks-to-raise-16-bln-via-asset-backed-securities-idUSL8N1R00DL,Technology News,Reuters,technology news,tech
1,-6487,-6487,2018-02-02 12:35:00,2018,2.0,2,Tierney McAfee,Anthony Scaramucci Dishes on His Firing in Vanity Fair Interview,"In a new interview with Vanity Fair, President Trump’s short-lived White House Communications Director Anthony Scaramucci recalls his 10 tumultuous days at ...",https://people.com/politics/anthony-scaramucci-firing-vanity-fair-interview/,politics,People,politics,politics
2,-905,-1118,2019-12-02 00:00:00,2019,12.0,2,None,Duncan Hunter expected to change plea to guilty in federal court,"Rep. Duncan Hunter, R-Ca, is expected to appear in federal court on Tuesday to plead guilty for spending more than $200,000 in campaign funds on personal ex...",https://www.cnbc.com/2019/12/02/duncan-hunter-expected-to-change-plea-to-guilty-in-federal-court.html,Politics,CNBC,politics,politics


Columns:
['idx', 'article_idx', 'date', 'year', 'month', 'day', 'author', 'title', 'article', 'url', 'section', 'publication', 'section_clean', 'meta_section']


## 5. Basic Column Validation

The model needs an article as input and a headline/title as target.

In [37]:
required_cols = ["article", "title"]
missing = [col for col in required_cols if col not in raw_df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}. Please rename your columns first.")

before_base = len(raw_df)
missing_mask = raw_df[["article", "title"]].isna().any(axis=1)
missing_rows = raw_df[missing_mask].copy()

model_base_df = raw_df.dropna(subset=["article", "title"]).copy()
model_base_df["article"] = model_base_df["article"].astype(str)
model_base_df["title"] = model_base_df["title"].astype(str)

after_base = len(model_base_df)
removed_base = before_base - after_base
removed_base_pct = (removed_base / before_base * 100) if before_base else 0.0

print("Why this step: article and title are mandatory because this is supervised headline generation.")
print("Rows before dropna:", before_base)
print("Rows after dropna :", after_base)
print(f"Removed rows      : {removed_base} ({removed_base_pct:.3f}%)")
if removed_base > 0:
    print("Sample removed rows with missing article/title:")
    show_cols = [c for c in ["article", "title", "url", "publication", "date"] if c in missing_rows.columns]
    display(missing_rows[show_cols].head(5))

Why this step: article and title are mandatory because this is supervised headline generation.
Rows before dropna: 55287
Rows after dropna : 55287
Removed rows      : 0 (0.000%)


## 6. Model-Safe Cleaning Functions

 **light transformer-level cleaning**, but with a bit stronger real-world cleanup for news data



For headline generation, we still preserve natural language as much as possible:

- Keep punctuation
- Keep numbers
- Keep capitalization
- Keep stopwords
- best not to stem
- best not to lemmatize
- best not to lowercase everything

What added here:

- handling for newswire prefixes (like `UPDATE 1-`, `CORRECTED`, location datelines)
- Removal of repetitive boilerplate lines in article bodies (`Reporting by ...`, `Editing by ...`, etc.)

Why?

Because these patterns are common in All The News dataset and can confuse the model, while not adding real semantic value for headline generation.

In [38]:
try:
    import contractions
    HAS_CONTRACTIONS = True
except Exception:
    HAS_CONTRACTIONS = False
    print("contractions library not installed. Using fallback contraction rules.")

try:
    from langdetect import detect_langs
    HAS_LANGDETECT = True
except Exception:
    HAS_LANGDETECT = False
    print("langdetect is not installed. Language filter will be skipped unless you install it.")

try:
    from transformers import AutoTokenizer
    HAS_TRANSFORMERS = True
except Exception:
    HAS_TRANSFORMERS = False
    print("transformers is not installed. Tokenizer-length filter will be skipped unless you install it.")

fallback_contractions = {
    "can't": "cannot", "won't": "will not", "n't": " not",
    "i'm": "i am", "it's": "it is", "he's": "he is", "she's": "she is",
    "that's": "that is", "there's": "there is", "what's": "what is",
    "you're": "you are", "we're": "we are", "they're": "they are",
    "i've": "i have", "we've": "we have", "they've": "they have",
    "i'll": "i will", "you'll": "you will", "we'll": "we will",
    "don't": "do not", "doesn't": "does not", "didn't": "did not",
    "isn't": "is not", "aren't": "are not", "wasn't": "was not",
    "weren't": "were not", "hasn't": "has not", "haven't": "have not",
    "hadn't": "had not", "wouldn't": "would not", "shouldn't": "should not",
    "couldn't": "could not"
}

# Broadened patterns to catch prefixes with colons, dashes, or just trailing spaces
NEWSWIRE_PREFIX_PATTERNS = [
    r"^\s*(?:REFILE-)?(?:UPDATE|CORRECTED|WRAPUP|PREVIEW)(?:\s+\d+)?\s*[:\-]\s*", #matches: UPDATE 2: , UPDATE 1 - , REFILE-UPDATE 3: , CORRECTED: , WRAPUP 4 - , PREVIEW:
    r"^\s*(?:UPDATE|PREVIEW|BRIEF)\b\s*[:\-]?\s*", #matches: UPDATE , PREVIEW - , BRIEF: (this pattern is broader than the previous one)
    r"^\s*\([A-Za-z]+\)\s*-\s*", #matches: (Reuters) - , (AP) -
    r"^\s*[A-Z][A-Za-z\s\.'/\-]{2,80}\s*\([A-Za-z\.\s]+\)\s*-\s*", #matches: WASHINGTON (Reuters) - , New York (AP) - , LONDON (Reuters) -
    #if the above ones missed the update, brief, refile ... words, i stated them to be matched in whatever form when they appear in the start of the article
    r"^\s*brief\b",
    r"^\s*update\s*\d*\s*[-:]",
    r"^\s*refile\s*-?\s*update\s*\d*\s*[-:]",
    r"^\s*wrapup\s*\d*\s*[-:]",
    r"^\s*press digest\b",
    r"^\s*factors to watch\b",
    r"^\s*preview\b",
    r"^\s*team by team analysis\b",
    r"^\s*pro rata podcast\b",
    r"^\s*us stocks snapshot\b",
    r"^\s*rising:\s+[A-Za-z]+\s+\d{1,2},\s+\d{4}\s*\|\s*TheHill\s*$" #matches: Rising: April 15, 2019 | TheHill , Rising: May 3, 2020 | TheHill
]

BOILERPLATE_LINE_PATTERNS = [
    r"^\s*Reporting by\b",
    r"^\s*Additional reporting by\b",
    r"^\s*Editing by\b",
    r"^\s*Compiled by\b",
    r"^\s*For related stories\b",
    r"^\s*Click here\b",
    r"^\s*Follow us\b",
    r"^\s*Subscribe\b",
    r"^\s*\(This story has been corrected.*\)$",
    r"^\s*\(Reuters\)\s*-\s*$"
]

combined_boilerplate_line_pattern = re.compile(
    "|".join(BOILERPLATE_LINE_PATTERNS),
    flags=re.IGNORECASE
)

def normalize_unicode(text):
    return unicodedata.normalize("NFKC", str(text))

def remove_html_entities_and_tags(text):
    text = html.unescape(str(text))
    text = re.sub(r"<[^>]+>", " ", text)
    return text

def normalize_quotes_dashes(text):
    replacements = {"“": '"', "”": '"', "‘": "'", "’": "'", "—": " - ", "–": " - ", "…": "..."}
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text

def remove_urls_and_emails(text):
    text = re.sub(r"http\S+|www\.\S+", " ", str(text))
    text = re.sub(r"\S+@\S+", " ", text)
    return text

def remove_newswire_prefixes(text):
    text = str(text)
    for pat in NEWSWIRE_PREFIX_PATTERNS:
        text = re.sub(pat, "", text, flags=re.IGNORECASE)
    return text

def strip_article_boilerplate(text):
    lines = re.split(r"\n+", str(text))
    cleaned_lines = [line.strip() for line in lines if line.strip() and not combined_boilerplate_line_pattern.search(line)]
    return "\n".join(cleaned_lines)

def fix_whitespace(text):
    text = str(text).replace(" ", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def normalize_title_punctuation(title):
    """Normalize spacing and repeated punctuation in headline targets only."""
    title = str(title).strip()
    title = re.sub(r"\s+", " ", title)
    title = re.sub(r"\s+([,.;:!?])", r"\1", title)
    title = re.sub(r"([,.;:!?]){2,}", r"\1", title)
    return title.strip()

def expand_contractions_text(text):
    text = str(text)
    if HAS_CONTRACTIONS:
        return contractions.fix(text)
    for c, e in fallback_contractions.items():
        text = re.sub(re.escape(c), e, text, flags=re.IGNORECASE)
    return text

def clean_for_transformer(text, expand_contractions_flag=False, is_title=False):
    text = normalize_unicode(text)
    text = remove_html_entities_and_tags(text)
    text = normalize_quotes_dashes(text)
    text = remove_urls_and_emails(text)
    text = remove_newswire_prefixes(text)

    # Prefix removal is essential for both body and title in news datasets
    text = remove_newswire_prefixes(text)

    if not is_title:
        text = strip_article_boilerplate(text)

    if expand_contractions_flag:
        text = expand_contractions_text(text)

    text = fix_whitespace(text)
    return text

## 7. Apply Transformer-Level Cleaning

This creates the two important clean text columns:

- `article_clean_transformer`
- `title_clean`


In [39]:
# True only if we want contractions like "don't" -> "do not"
# For news headline generation, False is usually safer because it preserves the original style
EXPAND_CONTRACTIONS = False

model_df = model_base_df.copy()

model_df["article_clean_transformer"] = model_df["article"].apply(
    lambda x: clean_for_transformer(x, expand_contractions_flag=EXPAND_CONTRACTIONS, is_title=False)
)
model_df["title_clean"] = model_df["title"].apply(
    lambda x: clean_for_transformer(x, expand_contractions_flag=EXPAND_CONTRACTIONS, is_title=True)
)

print("After cleaning:", model_df.shape)
display(model_df[["article", "article_clean_transformer", "title", "title_clean"]].head(3))


# Preprocessing audit utilities

INITIAL_ROWS = len(model_df)
preprocess_audit = []


def log_filter_step(step_name, reason, before_df, after_df, show_cols=None, sample_n=5):
    """Logging what was removed in this step + show a small sample for justification."""
    before_count = len(before_df)
    after_count = len(after_df)
    removed_count = before_count - after_count

    removed_idx = before_df.index.difference(after_df.index)
    removed_df = before_df.loc[removed_idx].copy()

    removed_pct_step = (removed_count / before_count * 100) if before_count else 0.0
    cumulative_removed = INITIAL_ROWS - after_count
    cumulative_removed_pct = (cumulative_removed / INITIAL_ROWS * 100) if INITIAL_ROWS else 0.0

    preprocess_audit.append({
        "step": step_name,
        "reason": reason,
        "before_rows": before_count,
        "after_rows": after_count,
        "removed_rows": removed_count,
        "removed_pct_step": round(removed_pct_step, 3),
        "cumulative_removed": cumulative_removed,
        "cumulative_removed_pct": round(cumulative_removed_pct, 3),
    })

    print("=" * 90)
    print(f"Step: {step_name}")
    print(f"Why this step: {reason}")
    print(f"Rows before: {before_count}")
    print(f"Rows after : {after_count}")
    print(f"Removed    : {removed_count} ({removed_pct_step:.3f}% of this step input)")
    print(f"Cumulative removed from start: {cumulative_removed} ({cumulative_removed_pct:.3f}%)")

    if removed_count > 0:
        print(f"Sample removed rows (up to {sample_n}):")
        if show_cols is None:
            show_cols = [c for c in ["title_clean", "article_clean_transformer"] if c in removed_df.columns]
        else:
            # title+article visibility when possible so outputs are easier to show
            show_cols = [c for c in show_cols if c in removed_df.columns]
            if "title_clean" in removed_df.columns and "title_clean" not in show_cols:
                show_cols.append("title_clean")
            if "article_clean_transformer" in removed_df.columns and "article_clean_transformer" not in show_cols:
                show_cols.append("article_clean_transformer")

        if show_cols:
            sample_view = removed_df[show_cols].head(sample_n).copy()
            # short previews so long article text is still readable in table form
            if "article_clean_transformer" in sample_view.columns:
                sample_view["article_preview"] = sample_view["article_clean_transformer"].astype(str).str.slice(0, 220)
            if "title_clean" in sample_view.columns:
                sample_view["title_preview"] = sample_view["title_clean"].astype(str).str.slice(0, 120)
            display(sample_view)
        else:
            display(removed_df.head(sample_n))
    else:
        print("No rows removed in this step.")


def log_skipped_step(step_name, reason, current_df):
    """Log skipped optional steps so the audit table is complete."""
    current_count = len(current_df)
    cumulative_removed = INITIAL_ROWS - current_count
    cumulative_removed_pct = (cumulative_removed / INITIAL_ROWS * 100) if INITIAL_ROWS else 0.0

    preprocess_audit.append({
        "step": step_name,
        "reason": reason,
        "before_rows": current_count,
        "after_rows": current_count,
        "removed_rows": 0,
        "removed_pct_step": 0.0,
        "cumulative_removed": cumulative_removed,
        "cumulative_removed_pct": round(cumulative_removed_pct, 3),
    })

    print("=" * 90)
    print(f"Step: {step_name}")
    print(f"Why this step: {reason}")
    print("Step status: skipped (no rows removed).")

After cleaning: (55287, 16)


,article,article_clean_transformer,title,title_clean
0,SHANGHAI (Reuters) - A unit of Chinese ride-hailing firm Didi Chuxing has submitted an application to raise 10 billion yuan ($1.6 billion) through an issuan...,A unit of Chinese ride-hailing firm Didi Chuxing has submitted an application to raise 10 billion yuan ($1.6 billion) through an issuance of asset-backed se...,China's Didi looks to raise $1.6 billion via asset-backed securities,China's Didi looks to raise $1.6 billion via asset-backed securities
1,"In a new interview with Vanity Fair, President Trump’s short-lived White House Communications Director Anthony Scaramucci recalls his 10 tumultuous days at ...","In a new interview with Vanity Fair, President Trump's short-lived White House Communications Director Anthony Scaramucci recalls his 10 tumultuous days at ...",Anthony Scaramucci Dishes on His Firing in Vanity Fair Interview,Anthony Scaramucci Dishes on His Firing in Vanity Fair Interview
2,"Rep. Duncan Hunter, R-Ca, is expected to appear in federal court on Tuesday to plead guilty for spending more than $200,000 in campaign funds on personal ex...","Rep. Duncan Hunter, R-Ca, is expected to appear in federal court on Tuesday to plead guilty for spending more than $200,000 in campaign funds on personal ex...",Duncan Hunter expected to change plea to guilty in federal court,Duncan Hunter expected to change plea to guilty in federal court


## 8. Remove Empty or Very Short Text After Cleaning

This is a safety check after cleaning.

Some rows may become empty or too short after URL/HTML/noise removal.

In [40]:
before_df = model_df.copy()

model_df = model_df[
    (model_df["article_clean_transformer"].str.len() > 100) &
    (model_df["title_clean"].str.len() > 5)
].copy()

log_filter_step(
    step_name="Safety char-length filter",
    reason="Remove rows that become too short after cleaning; very short body/title pairs are usually weak supervision.",
    before_df=before_df,
    after_df=model_df,
    show_cols=["title_clean", "article_clean_transformer"]
)

Step: Safety char-length filter
Why this step: Remove rows that become too short after cleaning; very short body/title pairs are usually weak supervision.
Rows before: 55287
Rows after : 55287
Removed    : 0 (0.000% of this step input)
Cumulative removed from start: 0 (0.000%)
No rows removed in this step.


## 9. Remove Duplicate Articles and Duplicate Input-Target Pairs

This prevents data leakage and repeated examples

The most important duplicate to remove is the duplicated `article_clean_transformer`, because the same article may appear with repeated or slightly different titles

In [41]:
before_df = model_df.copy()

# Showing clear duplicates before dropping.
dup_mask_article = before_df.duplicated(subset=["article_clean_transformer"], keep=False)
dup_evidence = before_df[dup_mask_article].copy()
if len(dup_evidence) > 0:
    dup_evidence["duplicate_group_size"] = dup_evidence.groupby("article_clean_transformer")["article_clean_transformer"].transform("size")
    print("Duplicate evidence before exact dedup (same article text appears multiple times):")
    display(
        dup_evidence[["title_clean", "article_clean_transformer", "duplicate_group_size"]]
        .sort_values("duplicate_group_size", ascending=False)
        .head(12)
    )
else:
    print("No exact article duplicates found before this step.")

model_df = model_df.drop_duplicates(subset=["article_clean_transformer"], keep="first").copy()
model_df = model_df.drop_duplicates(subset=["article_clean_transformer", "title_clean"], keep="first").copy()

log_filter_step(
    step_name="Exact duplicate removal",
    reason="Remove repeated article pairs to reduce memorization and data leakage across splits.",
    before_df=before_df,
    after_df=model_df,
    show_cols=["title_clean", "article_clean_transformer", "duplicate_group_size"]
)

Duplicate evidence before exact dedup (same article text appears multiple times):


,title_clean,article_clean_transformer,duplicate_group_size
269,Friend of U.S. high court nominee asks not to speak publicly on alleged assault,"Mark Judge, a man identified as a witness to an alleged sexual assault by U.S. Supreme Court nominee Brett Kavanaugh, said on Tuesday he does not want to sp...",2
429,Facebook signs agreement with Washington state to end discriminatory ad targeting,Facebook Inc (FB.O) has signed an agreement with the state of Washington to stop third-party advertisers in the United States from excluding protected group...,2
704,"U.S. SEC chair grilled by Senate panel over cyber breach, Equifax",The chairman of the U.S. Securities and Exchange Commission (SEC) told a congressional committee on Tuesday he did not believe his predecessor Mary Jo White...,2
868,U.S. Senate intelligence panel subpoenas Trump's son: sources,"The U.S. Senate Intelligence Committee has subpoenaed one of the president's sons, Donald Trump Jr., to answer questions about his contacts with Russia, two...",2
1125,Monsanto patent victory seen as a boost for biotech investment in India,"India's Supreme Court ruled on Tuesday that Monsanto can claim patents on its genetically modified (GM) cotton seeds, a victory for the U.S. company that is...",2
1371,U.S. returns bells looted after Philippine wartime massacre,"Church bells taken as war trophies by U.S. forces more than a century ago arrived in the Philippines on Tuesday, ending Manila's decades-long quest for the ...",2
1514,U.S. Supreme Court weighs Microsoft overseas data fight,"A major privacy rights fight between Microsoft Corp and the Justice Department reaches the Supreme Court this week, with the justices considering whether U....",2
2383,U.S. Supreme Court declines billionaire appeal in California beach access suit,"The U.S. Supreme Court handed a victory to proponents of open access to California's coastline on Monday, declining the appeal of a billionaire venture capi...",2
2814,"President of the Mormon church, Thomas Monson, dies at 90","The leader of the Mormon church, Thomas Monson, has died at his home in Salt Lake City, Utah, the church said on Wednesday. He was 90. Monson became the 16t...",2
2898,Trumps says conservative House lawmakers line up behind healthcare bill,U.S. President Donald Trump on Friday met with a dozen conservative Republicans from the House of Representatives and said all of them were now planning to ...,2


Step: Exact duplicate removal
Why this step: Remove repeated article pairs to reduce memorization and data leakage across splits.
Rows before: 55287
Rows after : 55166
Removed    : 121 (0.219% of this step input)
Cumulative removed from start: 121 (0.219%)
Sample removed rows (up to 5):


,title_clean,article_clean_transformer,article_preview,title_preview
5021,U.S. Senate intelligence panel subpoenas Trump's son: sources,"The U.S. Senate Intelligence Committee has subpoenaed one of the president's sons, Donald Trump Jr., to answer questions about his contacts with Russia, two...","The U.S. Senate Intelligence Committee has subpoenaed one of the president's sons, Donald Trump Jr., to answer questions about his contacts with Russia, two...",U.S. Senate intelligence panel subpoenas Trump's son: sources
6397,Monsanto patent victory seen as a boost for biotech investment in India,"India's Supreme Court ruled on Tuesday that Monsanto can claim patents on its genetically modified (GM) cotton seeds, a victory for the U.S. company that is...","India's Supreme Court ruled on Tuesday that Monsanto can claim patents on its genetically modified (GM) cotton seeds, a victory for the U.S. company that is...",Monsanto patent victory seen as a boost for biotech investment in India
7042,Samsung Electronics delays Galaxy Fold media events in China,"Smartphone maker Samsung Electronics Co Ltd has postponed media events for its Galaxy Fold planned for this week in Hong Kong and Shanghai, a company offici...","Smartphone maker Samsung Electronics Co Ltd has postponed media events for its Galaxy Fold planned for this week in Hong Kong and Shanghai, a company offici...",Samsung Electronics delays Galaxy Fold media events in China
9094,Republicans grapple with whether to back Trump for White House,"Republican lawmakers, operatives and donors grappled with whether to support Donald Trump, who effectively clinched his party's presidential nomination this...","Republican lawmakers, operatives and donors grappled with whether to support Donald Trump, who effectively clinched his party's presidential nomination this...",Republicans grapple with whether to back Trump for White House
10323,U.S. investigating Cambridge Analytica: New York Times,"The U.S. Justice Department and the FBI are investigating Cambridge Analytica, a now-defunct political data firm embroiled in a scandal over its handling of...","The U.S. Justice Department and the FBI are investigating Cambridge Analytica, a now-defunct political data firm embroiled in a scandal over its handling of...",U.S. investigating Cambridge Analytica: New York Times


## 10. Extra Dedup Step (Loose Near-Duplicate Key)

In news datasets, removing only exact duplicates is usually not enough

The same story can appear more than once with very small edits, like extra spaces, punctuation differences, or short wire-service updates. Because of this,a *loose* dedup key was added

The idea normalizing the article text strongly (lowercase, remove punctuation/noise, fix spaces) and use the first part of the text to build a key. If two rows get the same loose key, they are treated as near-duplicates, and only one of them is kept

This helps reduce repetition and possible leakage across splits, so the model learns from more diverse examples

This is still not semantic deduplication, but it is a practical and fast method that works well for Colab-sized preprocessing

In [42]:
def build_loose_dedup_key(text, max_chars=700):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text[:max_chars]

before_df = model_df.copy()
before_df["dedup_key_loose"] = before_df["article_clean_transformer"].apply(build_loose_dedup_key)

# Show evidence for loose near-duplicate groups before dropping.
loose_dup_mask = before_df.duplicated(subset=["dedup_key_loose"], keep=False)
loose_evidence = before_df[loose_dup_mask].copy()
if len(loose_evidence) > 0:
    loose_evidence["loose_group_size"] = loose_evidence.groupby("dedup_key_loose")["dedup_key_loose"].transform("size")
    print("Loose near-duplicate evidence before dedup (same normalized key appears multiple times):")
    display(
        loose_evidence[["title_clean", "article_clean_transformer", "dedup_key_loose", "loose_group_size"]]
        .sort_values("loose_group_size", ascending=False)
        .head(12)
    )
else:
    print("No loose near-duplicate groups found before this step.")

model_df["dedup_key_loose"] = model_df["article_clean_transformer"].apply(build_loose_dedup_key)
model_df = model_df.drop_duplicates(subset=["dedup_key_loose"], keep="first").copy()

log_filter_step(
    step_name="Loose near-duplicate removal",
    reason="Catch practical near-duplicates with tiny edits so the model sees more diverse examples.",
    before_df=before_df,
    after_df=model_df,
    show_cols=["title_clean", "article_clean_transformer", "dedup_key_loose", "loose_group_size"]
)

Loose near-duplicate evidence before dedup (same normalized key appears multiple times):


,title_clean,article_clean_transformer,dedup_key_loose,loose_group_size
6716,"Judge orders State Department to review 14,900 Clinton emails","A judge ordered the U.S. State Department on Monday to review for possible release 14,900 of Hillary Clinton's emails and attachments that the FBI found whe...",a judge ordered the u s state department on monday to review for possible release 14 900 of hillary clinton s emails and attachments that the fbi found when...,3
19826,"Judge orders State Department to review 14,900 Clinton emails","A judge ordered the U.S. State Department on Monday to review for possible release 14,900 of Hillary Clinton's emails and attachments that the FBI found whe...",a judge ordered the u s state department on monday to review for possible release 14 900 of hillary clinton s emails and attachments that the fbi found when...,3
51206,"Judge orders State Department to review 14,900 Clinton emails","A judge ordered the U.S. State Department on Monday to review for possible release 14,900 of Hillary Clinton's emails and attachments that the FBI found whe...",a judge ordered the u s state department on monday to review for possible release 14 900 of hillary clinton s emails and attachments that the fbi found when...,3
1113,"Pennsylvania governor raises minimum wage for state workers, contractors","HARRISBURG, Pa. (Reuters) - Pennsylvania Governor Tom Wolf raised the minimum wage for state workers and employees of some contractors by 40 percent to $10....",harrisburg pa reuters pennsylvania governor tom wolf raised the minimum wage for state workers and employees of some contractors by 40 percent to 10 15 an h...,2
821,"U.S. budget deal sets up wider fight over deficits, immigration","The budget deal passed by U.S lawmakers early on Friday will alleviate the spending fights that marked President Donald Trump's first year in office, but se...",the budget deal passed by u s lawmakers early on friday will alleviate the spending fights that marked president donald trump s first year in office but set...,2
1290,"Prior to his SEC nomination, Clayton communicated with SEC contractor","Before Wall Street attorney Jay Clayton was nominated to be head of the U.S. Securities and Exchange Commission, he communicated with more than a half dozen...",before wall street attorney jay clayton was nominated to be head of the u s securities and exchange commission he communicated with more than a half dozen o...,2
1586,US Supreme Court ruling threatens massive talc litigation against J&J,Johnson & Johnson is seizing upon a U.S. Supreme Court ruling from Monday limiting where injury lawsuits can be filed to fight off claims it failed to warn ...,johnson johnson is seizing upon a u s supreme court ruling from monday limiting where injury lawsuits can be filed to fight off claims it failed to warn wom...,2
2554,Family of man shot by Cincinnati university officer sues: report,The family of a black man shot and killed last summer by a University of Cincinnati police officer during a traffic stop sued the officer in federal court o...,the family of a black man shot and killed last summer by a university of cincinnati police officer during a traffic stop sued the officer in federal court o...,2
1002,Exclusive: U.S. group Sierra Club seeks probe of EPA's Pruitt over CO2 comments,"U.S. environmental group the Sierra Club has asked the Environmental Protection Agency's inspector general to investigate whether the agency's head, Scott P...",u s environmental group the sierra club has asked the environmental protection agency s inspector general to investigate whether the agency s head scott pru...,2
2811,Congress probes N.Y. Fed's handling of Bangladesh Bank heist - letter,A U.S. congressional committee has launched a probe into the Federal Reserve Bank of New York's handling of the cyber theft of $81 million from one of its a...,a u s congressional committee has launched a probe into the federal reserve bank of new york s handling of the cybe

Step: Loose near-duplicate removal
Why this step: Catch practical near-duplicates with tiny edits so the model sees more diverse examples.
Rows before: 55166
Rows after : 55041
Removed    : 125 (0.227% of this step input)
Cumulative removed from start: 246 (0.445%)
Sample removed rows (up to 5):


,title_clean,article_clean_transformer,dedup_key_loose,article_preview,title_preview
4170,Family of man shot by Cincinnati university officer sues-report,The family of a black man shot and killed last summer by a University of Cincinnati police officer during a traffic stop sued the officer in federal court o...,the family of a black man shot and killed last summer by a university of cincinnati police officer during a traffic stop sued the officer in federal court o...,The family of a black man shot and killed last summer by a University of Cincinnati police officer during a traffic stop sued the officer in federal court o...,Family of man shot by Cincinnati university officer sues-report
8873,EPA official resigns over Flint water crisis,"The head of the Midwest region of the U.S. Environmental Protection Agency offered her resignation over the water contamination crisis in Flint, Michigan, t...",the head of the midwest region of the u s environmental protection agency offered her resignation over the water contamination crisis in flint michigan the ...,"The head of the Midwest region of the U.S. Environmental Protection Agency offered her resignation over the water contamination crisis in Flint, Michigan, t...",EPA official resigns over Flint water crisis
9890,YouTube's new plan to deal with awful comments: Have commenters help moderate,YouTube has a new plan to start turning around its famously bad comments: get the community involved in moderating them. It launched a new program this week...,youtube has a new plan to start turning around its famously bad comments get the community involved in moderating them it launched a new program this week c...,YouTube has a new plan to start turning around its famously bad comments: get the community involved in moderating them. It launched a new program this week...,YouTube's new plan to deal with awful comments: Have commenters help moderate
10352,Trump order paves way for agencies to weaken health law,"President Donald Trump is ordering federal agencies to undermine Obamacare through regulatory action, a move that could weaken enforcement of the requiremen...",president donald trump is ordering federal agencies to undermine obamacare through regulatory action a move that could weaken enforcement of the requirement...,"President Donald Trump is ordering federal agencies to undermine Obamacare through regulatory action, a move that could weaken enforcement of the requiremen...",Trump order paves way for agencies to weaken health law
11480,U.S. charges three people in $1 bln Medicare fraud scheme,"The U.S. Department of Justice unveiled its largest-ever criminal healthcare fraud case against individuals on Friday, charging the owner of Miami-based ass...",the u s department of justice unveiled its largest ever criminal healthcare fraud case against individuals on friday charging the owner of miami based assis...,"The U.S. Department of Justice unveiled its largest-ever criminal healthcare fraud case against individuals on Friday, charging the owner of Miami-based ass...",U.S. charges three people in $1 bln Medicare fraud scheme


### 10.1 Embedding-Based Near-Duplicate Removal

This catches articles that are semantically very similar even when they are not exact or loose text duplicates.

It is optional. Turn it on after the normal pipeline runs successfully.

In [43]:
USE_EMBEDDING_NEAR_DUPLICATE_FILTER = True
EMBEDDING_NEAR_DUP_THRESHOLD = 0.94
EMBEDDING_DEDUP_MODEL_NAME = "all-MiniLM-L6-v2" #lightweight sentence-transformer model that converts text into imbeddings

if USE_EMBEDDING_NEAR_DUPLICATE_FILTER:
    try:
        from sentence_transformers import SentenceTransformer
        from sklearn.neighbors import NearestNeighbors
        HAS_EMBEDDING_DEDUP = True
    except Exception:
        HAS_EMBEDDING_DEDUP = False
        print("sentence-transformers or sklearn NearestNeighbors missing. Embedding dedup will be skipped.")

    if HAS_EMBEDDING_DEDUP:
        print("Building embeddings for near-duplicate detection...")
        emb_model = SentenceTransformer(EMBEDDING_DEDUP_MODEL_NAME)

        # Use first 1500 chars because lead text usually identifies the news story well
        dedup_texts = model_df["article_clean_transformer"].astype(str).str[:1500].tolist()
        embeddings = emb_model.encode(
            dedup_texts,
            normalize_embeddings=True,
            show_progress_bar=True,
            batch_size=64
        )

        # Cosine similarity threshold becomes cosine distance radius = 1 - similarity (1 - 0.94 = 0.06)
        radius = 1 - EMBEDDING_NEAR_DUP_THRESHOLD
        nn = NearestNeighbors(metric="cosine", radius=radius, algorithm="brute")
        nn.fit(embeddings)
        neighbors = nn.radius_neighbors(embeddings, return_distance=False)

        keep_positions = []
        removed_positions = set()

        for i, neigh in enumerate(neighbors):
            if i in removed_positions:
                continue
            keep_positions.append(i)
            for j in neigh:
                if j != i:
                    removed_positions.add(int(j))

        before_df = model_df.copy()
        model_df = model_df.iloc[keep_positions].copy()

        log_filter_step(
            step_name="Embedding-based near-duplicate removal",
            reason="Remove semantically similar duplicate stories that text-based dedup may miss.",
            before_df=before_df,
            after_df=model_df,
            show_cols=["title_clean", "article_clean_transformer"]
        )
else:
    log_skipped_step(
        step_name="Embedding-based near-duplicate removal",
        reason="Skipped by default because it can be slow; set USE_EMBEDDING_NEAR_DUPLICATE_FILTER=True to enable it.",
        current_df=model_df
    )


Building embeddings for near-duplicate detection...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/861 [00:00<?, ?it/s]

Step: Embedding-based near-duplicate removal
Why this step: Remove semantically similar duplicate stories that text-based dedup may miss.
Rows before: 55041
Rows after : 54902
Removed    : 139 (0.253% of this step input)
Cumulative removed from start: 385 (0.696%)
Sample removed rows (up to 5):


,title_clean,article_clean_transformer,article_preview,title_preview
9672,"Fearful of bias, Google blocks gender-based pronouns from new AI tool","Alphabet Inc's (GOOGL.O) Google in May introduced a slick feature for Gmail that automatically completes sentences for users as they type. Tap out ""I love"" ...","Alphabet Inc's (GOOGL.O) Google in May introduced a slick feature for Gmail that automatically completes sentences for users as they type. Tap out ""I love"" ...","Fearful of bias, Google blocks gender-based pronouns from new AI tool"
11683,Republican strategy on healthcare bill in flux ahead of vote,"Republican Senate leaders aim to hold a procedural vote as early as Tuesday to take up legislation to repeal or replace Obamacare, but it remained unclear w...","Republican Senate leaders aim to hold a procedural vote as early as Tuesday to take up legislation to repeal or replace Obamacare, but it remained unclear w...",Republican strategy on healthcare bill in flux ahead of vote
11853,Trump officials say U.S.-China trade talks to resume next week,Top representatives of the United States and China are organizing a resumption of talks for next week to try to resolve a year-long trade war between the wo...,Top representatives of the United States and China are organizing a resumption of talks for next week to try to resolve a year-long trade war between the wo...,Trump officials say U.S.-China trade talks to resume next week
12486,"Facebook critics want regulation, investigation after data misuse",Facebook Inc faced new calls for regulation from within U.S. Congress and was hit with questions about personal data safeguards on Saturday after reports a ...,Facebook Inc faced new calls for regulation from within U.S. Congress and was hit with questions about personal data safeguards on Saturday after reports a ...,"Facebook critics want regulation, investigation after data misuse"
13349,"Intel, John McAfee settle lawsuits over antivirus pioneer's name","John McAfee, the creator of eponymous antivirus computer software, has settled a lawsuit against Intel over his right to use his name on other projects afte...","John McAfee, the creator of eponymous antivirus computer software, has settled a lawsuit against Intel over his right to use his name on other projects afte...","Intel, John McAfee settle lawsuits over antivirus pioneer's name"


## 11. Word Count Checks for Model Filtering

This part filters the actual data that will go to the model.

rules for a headline generation task:

- Article/input should not be too short
- Headline/target should not be too short
- Headline/target should not be too long

In [44]:
def simple_word_count(text):
    return len(str(text).split())

model_df["input_word_count"] = model_df["article_clean_transformer"].apply(simple_word_count)
model_df["target_word_count"] = model_df["title_clean"].apply(simple_word_count)

print("Input/article word counts:")
display(model_df["input_word_count"].describe())

print("Target/headline word counts:")
display(model_df["target_word_count"].describe())

Input/article word counts:


,input_word_count
count,54902.000000
mean,416.227351
std,231.715976
min,46.000000
25%,242.000000
50%,391.000000
75%,565.000000
max,1016.000000


Target/headline word counts:


,target_word_count
count,54902.000000
mean,10.070125
std,2.252248
min,3.000000
25%,9.000000
50%,10.000000
75%,11.000000
max,20.000000


## 12. Final Length Filtering (Word + tokenizer-level policy)

original word-based thresholds is kept because they are simple and stable

an optional tokenizer-based filter is added, because transformer models care about token count, not just word count

 Why this step:
 - Word limits are simple and good for first cleanup.
 - So we keep word filtering, then apply model-aware token handling:
 - truncate long inputs to model max length
 - drop only extreme outliers
 - keep realistic target token lengths

In [45]:
MIN_INPUT_WORDS = 50
MIN_TARGET_WORDS = 3
MAX_TARGET_WORDS = 20

before_df = model_df.copy()

model_df = model_df[
    (model_df["input_word_count"] >= MIN_INPUT_WORDS) &
    (model_df["target_word_count"] >= MIN_TARGET_WORDS) &
    (model_df["target_word_count"] <= MAX_TARGET_WORDS)
].copy()

log_filter_step(
    step_name="Word-length filter",
    reason="Keep rows where article has enough context and headline length looks realistic for training.",
    before_df=before_df,
    after_df=model_df,
    show_cols=["title_clean", "input_word_count", "target_word_count"]
)


# Model-agnostic tokenizer policy

MODEL_CHECKPOINT = "google/flan-t5-base"  # other options: t5-base, bart-base, gpt2, etc...
USE_TOKENIZER_LENGTH_POLICY = True

# Defaults if model is not listed in overrides.
DEFAULT_MAX_INPUT_TOKENS = 512
DEFAULT_MIN_TARGET_TOKENS = 3
DEFAULT_MAX_TARGET_TOKENS = 32

# Only drop very extreme long inputs (after this value)
# Keep it larger than MAX_INPUT_TOKENS because normal long rows will be truncated, not dropped
HARD_DROP_INPUT_TOKENS = 1400

# Optional per-model overrides
MODEL_TOKEN_LIMITS = {
    "google/flan-t5-small": {"max_input": 512, "max_target": 32},
    "google/flan-t5-base":  {"max_input": 512, "max_target": 48},
    "t5-small":             {"max_input": 512, "max_target": 32},
    "t5-base":              {"max_input": 512, "max_target": 48},
    "facebook/bart-base":   {"max_input": 1024, "max_target": 64},
    #because GPT-2 has a total context limit of around 1024
    #so we should leave space for the generated headline
    #(960 input tokens + 64 generated tokens = 1024 total tokens)
    "gpt2":                 {"max_input": 960, "max_target": 64},
}

cfg = MODEL_TOKEN_LIMITS.get(MODEL_CHECKPOINT, {})
MAX_INPUT_TOKENS = cfg.get("max_input", DEFAULT_MAX_INPUT_TOKENS)
MIN_TARGET_TOKENS = DEFAULT_MIN_TARGET_TOKENS
MAX_TARGET_TOKENS = cfg.get("max_target", DEFAULT_MAX_TARGET_TOKENS)

if USE_TOKENIZER_LENGTH_POLICY and HAS_TRANSFORMERS:
    TOKENIZER_NAME = MODEL_CHECKPOINT
    print(f"Applying tokenizer-length policy with: {TOKENIZER_NAME}")

    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME, use_fast=True)

    # Avoiding noisy warnings while we inspect long sequence lengths
    tokenizer.model_max_length = 10_000

    # Count raw token lengths first
    model_df["input_token_count_raw"] = model_df["article_clean_transformer"].apply(
        lambda x: len(tokenizer.encode(str(x), add_special_tokens=True))
    )
    model_df["target_token_count"] = model_df["title_clean"].apply(
        lambda x: len(tokenizer.encode(str(x), add_special_tokens=True))
    )

    # Mark rows that exceed model input limit
    # NOT mutating article_clean_transformer here
    # keeping full cleaned text and apply truncation later per model alias
    model_df["was_input_truncated"] = model_df["input_token_count_raw"] > MAX_INPUT_TOKENS
    trunc_count = int(model_df["was_input_truncated"].sum())
    print(f"Rows above {MAX_INPUT_TOKENS} input tokens (will be truncated later per model): {trunc_count}")

    # For observing: capped token count as if truncated at MAX_INPUT_TOKENS
    model_df["input_token_count"] = model_df["input_token_count_raw"].clip(upper=MAX_INPUT_TOKENS)

    # Final token-level filtering:
    # dropping only extreme input outliers
    # enforcing target token bounds
    before_tok_df = model_df.copy()
    model_df = model_df[
        (model_df["input_token_count_raw"] <= HARD_DROP_INPUT_TOKENS) &
        (model_df["target_token_count"] >= MIN_TARGET_TOKENS) &
        (model_df["target_token_count"] <= MAX_TARGET_TOKENS)
    ].copy()

    log_filter_step(
        step_name="Tokenizer policy (truncate long, drop extreme outliers)",
        reason=(
            "Word count is approximate; token counts match transformer behavior. "
            "We keep more data by truncating normal long inputs and dropping only extreme outliers."
        ),
        before_df=before_tok_df,
        after_df=model_df,
        show_cols=[
            "title_clean",
            "input_token_count_raw",
            "input_token_count",
            "target_token_count",
            "was_input_truncated"
        ]
    )

else:
    log_skipped_step(
        step_name="Tokenizer policy (truncate long, drop extreme outliers)",
        reason="Skipped because transformers is missing or USE_TOKENIZER_LENGTH_POLICY=False.",
        current_df=model_df
    )

print("Word-count stats after filtering:")
display(model_df[["input_word_count", "target_word_count"]].describe())

if "input_token_count_raw" in model_df.columns:
    print("Raw input token stats (before truncation):")
    display(model_df["input_token_count_raw"].describe())

if "input_token_count" in model_df.columns:
    print("Final input token stats (after truncation):")
    display(model_df["input_token_count"].describe())

Step: Word-length filter
Why this step: Keep rows where article has enough context and headline length looks realistic for training.
Rows before: 54902
Rows after : 54874
Removed    : 28 (0.051% of this step input)
Cumulative removed from start: 413 (0.747%)
Sample removed rows (up to 5):


,title_clean,input_word_count,target_word_count,article_clean_transformer,article_preview,title_preview
388,U.S. Fed chief to discuss economy before House panel on February 27,47,12,"U.S. Federal Reserve Chairman Jerome Powell to testify before the House Financial Services Committee on Feb. 27, the committee's Democratic chairwoman said ...","U.S. Federal Reserve Chairman Jerome Powell to testify before the House Financial Services Committee on Feb. 27, the committee's Democratic chairwoman said ...",U.S. Fed chief to discuss economy before House panel on February 27
5451,"Person believed to have been stabbed in London Bridge area, suspect shot: source",49,13,"It is believed somebody was stabbed in the London Bridge area of central London on Friday and police have shot a suspect, a security source told Reuters on ...","It is believed somebody was stabbed in the London Bridge area of central London on Friday and police have shot a suspect, a security source told Reuters on ...","Person believed to have been stabbed in London Bridge area, suspect shot: source"
5590,Oman suspends Italian tourism flights to its Salala airport,48,9,"Oman has suspended Italian tourism flights to its Salala airport for a month to curb the spread of the coronavirus, the civil aviation authority said on Sun...","Oman has suspended Italian tourism flights to its Salala airport for a month to curb the spread of the coronavirus, the civil aviation authority said on Sun...",Oman suspends Italian tourism flights to its Salala airport
6692,Republican Senator Collins 'leaning against' new healthcare bill: AP,47,9,"Republican Senator Susan Collins has said she is ""leaning against"" the latest Republican healthcare bill, the Associated Press reported on Friday. The Maine...","Republican Senator Susan Collins has said she is ""leaning against"" the latest Republican healthcare bill, the Associated Press reported on Friday. The Maine...",Republican Senator Collins 'leaning against' new healthcare bill: AP
9029,Bolivia Armed Forces say it will protect 'essential public services',49,10,Bolivia's Armed Forces said in a statement on Monday that it would launch plans to protect essential public services amid escalating violence in the South A...,Bolivia's Armed Forces said in a statement on Monday that it would launch plans to protect essential public services amid escalating violence in the South A...,Bolivia Armed Forces say it will protect 'essential public services'


Applying tokenizer-length policy with: google/flan-t5-base
Rows above 512 input tokens (will be truncated later per model): 29808
Step: Tokenizer policy (truncate long, drop extreme outliers)
Why this step: Word count is approximate; token counts match transformer behavior. We keep more data by truncating normal long inputs and dropping only extreme outliers.
Rows before: 54874
Rows after : 54618
Removed    : 256 (0.467% of this step input)
Cumulative removed from start: 669 (1.210%)
Sample removed rows (up to 5):


,title_clean,input_token_count_raw,input_token_count,target_token_count,was_input_truncated,article_clean_transformer,article_preview,title_preview
292,"In bloodied Mexico, ambivalence, hope over amnesty proposal",1479,512,18,True,next Image 1 of 2 prev Image 2 of 2 MEXICO CITY - Lucia Diaz and other volunteers have found more than 300 bodies in clandestine graves along Mexico&aposs G...,next Image 1 of 2 prev Image 2 of 2 MEXICO CITY - Lucia Diaz and other volunteers have found more than 300 bodies in clandestine graves along Mexico&aposs G...,"In bloodied Mexico, ambivalence, hope over amnesty proposal"
377,Bannon role in Trump administration sets off critical firestorm,1425,512,12,True,"White supremacists and neo-Nazis have rarely, if ever, in recent history been so enthusiastic about a presidential appointment as Donald Trump's choice of S...","White supremacists and neo-Nazis have rarely, if ever, in recent history been so enthusiastic about a presidential appointment as Donald Trump's choice of S...",Bannon role in Trump administration sets off critical firestorm
539,"Lured by Early Warm Weather, Scorpions Emerge to Swarm Arizona Homes",1518,512,18,True,"SCOTTSDALE, Ariz. - The scorpions that scurry around this desert region emerged from their winter slumber early this year. Usually dormant until late March,...","SCOTTSDALE, Ariz. - The scorpions that scurry around this desert region emerged from their winter slumber early this year. Usually dormant until late March,...","Lured by Early Warm Weather, Scorpions Emerge to Swarm Arizona Homes"
825,What was in the DNC email leak?,1450,512,10,True,"(CNN)Debbie Wasserman Schultz's stewardship of the Democratic National Committee has been under fire through most of the presidential primary process. Now, ...","(CNN)Debbie Wasserman Schultz's stewardship of the Democratic National Committee has been under fire through most of the presidential primary process. Now, ...",What was in the DNC email leak?
874,"Iceland Goes to Polls Amid Scandals, Disgust and Distrust",1440,512,19,True,"REYKJAVIK, Iceland - As Iceland headed to the polls on Saturday to vote for members of one of the oldest Parliaments in the world, the shadow of political s...","REYKJAVIK, Iceland - As Iceland headed to the polls on Saturday to vote for members of one of the oldest Parliaments in the world, the shadow of political s...","Iceland Goes to Polls Amid Scandals, Disgust and Distrust"


Word-count stats after filtering:


,input_word_count,target_word_count
count,54618.000000,54618.000000
mean,413.821176,10.070306
std,229.024428,2.251316
min,50.000000,3.000000
25%,241.000000,9.000000
50%,390.000000,10.000000
75%,562.000000,11.000000
max,1003.000000,20.000000


Raw input token stats (before truncation):


,input_token_count_raw
count,54618.000000
mean,578.524589
std,317.744420
min,63.000000
25%,337.000000
50%,542.000000
75%,785.000000
max,1400.000000


Final input token stats (after truncation):


,input_token_count
count,54618.000000
mean,416.522099
std,136.354301
min,63.000000
25%,337.000000
50%,512.000000
75%,512.000000
max,512.000000


## 13. Headline Quality Filters

Some titles are still noisy even after basic cleaning.

Here applying simple quality heuristics to remove targets that are usually bad labels for training:

- too much punctuation/symbols
- too many digits
- all-caps shouting titles
- generic placeholders like "video", "live updates", etc.

In [46]:
GENERIC_BAD_TITLES = {
    "video", "photos", "live updates", "live blog", "breaking", "update", "brief"
}

def title_quality_metrics(title):
    t = str(title).strip()
    if not t:
        return pd.Series({
            "title_is_bad_generic": True,
            "title_punct_ratio": 1.0,
            "title_digit_ratio": 1.0,
            "title_upper_ratio": 1.0
        })

    letters = [c for c in t if c.isalpha()]
    digits = [c for c in t if c.isdigit()]
    punct = [c for c in t if not c.isalnum() and not c.isspace()]

    letter_count = max(len(letters), 1)
    char_count = max(len(t.replace(" ", "")), 1)

    upper_ratio = sum(1 for c in letters if c.isupper()) / letter_count
    digit_ratio = len(digits) / char_count
    punct_ratio = len(punct) / char_count

    low = t.lower()
    is_bad_generic = low in GENERIC_BAD_TITLES

    return pd.Series({
        "title_is_bad_generic": is_bad_generic,
        "title_punct_ratio": punct_ratio,
        "title_digit_ratio": digit_ratio,
        "title_upper_ratio": upper_ratio
    })

quality_df = model_df["title_clean"].apply(title_quality_metrics)
model_df = pd.concat([model_df, quality_df], axis=1)

before_df = model_df.copy()
model_df = model_df[
    (~model_df["title_is_bad_generic"]) &
    (model_df["title_punct_ratio"] <= 0.35) &
    (model_df["title_digit_ratio"] <= 0.40) &
    (model_df["title_upper_ratio"] <= 0.90)
].copy()

log_filter_step(
    step_name="Headline quality filter",
    reason="Remove headline targets that look noisy or low-information (generic labels, punctuation-heavy, etc.).",
    before_df=before_df,
    after_df=model_df,
    show_cols=["title_clean", "title_is_bad_generic", "title_punct_ratio", "title_digit_ratio", "title_upper_ratio"]
)

Step: Headline quality filter
Why this step: Remove headline targets that look noisy or low-information (generic labels, punctuation-heavy, etc.).
Rows before: 54618
Rows after : 54617
Removed    : 1 (0.002% of this step input)
Cumulative removed from start: 670 (1.212%)
Sample removed rows (up to 5):


,title_clean,title_is_bad_generic,title_punct_ratio,title_digit_ratio,title_upper_ratio,article_clean_transformer,article_preview,title_preview
37698,O.J. SIMPSON TRIAL 25 YEARS LATER NICOLE BROWN CHANGED DOMESTIC ABUSE,False,0.033898,0.033898,1.0,"Twenty-five years ago this month, Nicole Brown Simpson and her friend Ronald Goldman were killed in a knife attack outside her Brentwood home. From the star...","Twenty-five years ago this month, Nicole Brown Simpson and her friend Ronald Goldman were killed in a knife attack outside her Brentwood home. From the star...",O.J. SIMPSON TRIAL 25 YEARS LATER NICOLE BROWN CHANGED DOMESTIC ABUSE


In [47]:
# Removing news-source/location prefixes from article text
# "(CNN)Before his team..."
# "Washington (CNN)Former Veterans Affairs Secretary..."

NEWS_PREFIX_PATTERNS = [
    r"^\s*\(CNN\)\s*",                         # (CNN)
    r"^\s*[A-Z][A-Za-z\s.,'-]{1,40}\s*\(CNN\)\s*"  # Washington (CNN), New York (CNN), etc.
]

combined_news_prefix_pattern = re.compile(
    "|".join(NEWS_PREFIX_PATTERNS),
    flags=re.IGNORECASE
)

before_df = model_df.copy()

model_df["article_clean_transformer"] = (
    model_df["article_clean_transformer"]
    .astype(str)
    .str.replace(combined_news_prefix_pattern, "", regex=True)
    .str.strip()
)

print("Examples after removing CNN/location prefixes:")
display(
    model_df[["title_clean", "article_clean_transformer"]]
    .sample(10, random_state=42)
)

log_filter_step(
    step_name="CNN/source-prefix removal from articles",
    reason="Remove source/location prefixes such as '(CNN)' and 'Washington (CNN)' from article text without deleting the article row.",
    before_df=before_df,
    after_df=model_df,
    show_cols=["title_clean", "article_clean_transformer"]
)

Examples after removing CNN/location prefixes:


,title_clean,article_clean_transformer
26774,BOJ warns of cyber-attack vulnerability ahead of Olympic Games,"Japan's financial institutions must guard against cyber-attacks ahead of the 2020 Tokyo Olympic Games, with nearly 40% of banks and other firms experiencing..."
28210,"Texas woman arrested after attempting to strangle mother with kitchen knife, police say",Emily Collier has been charged with aggravated assault against a family member with a weapon. (Angelina County Judicial Records) A 23-year-old Texas woman w...
36019,EEOC Monitor: Digital intake system debuts in five cities,"Despite uncertainty about how the Equal Employment Opportunity Commission's priorities will shift under President Donald Trump, the agency is forging ahead ..."
47278,"Tesla's surge inspires fans to buy, skeptics to dig in, drives fear of missing out",Pretty much everyone on Wall Street has an opinion about Tesla (TSLA.O). The electric vehicle maker's stupendous rally in recent months has given shareholde...
14147,"This esports team raises $46 million, and investors include Will Smith",Esports organization Gen.G announced Wednesday it has raised $46 million in a funding round led by some big names in sports and entertainment. The latest in...
40067,Here's why Democrats face a tough Senate campaign this year,"Republicans are wasting no time getting ready for the 2018 Senate campaign, as several outside GOP groups have raised a record amount of money for the upcom..."
1613,Who is John Kasich?,"Since entering the race last July, Ohio Gov. John Kasich has taken something of a different approach from the other GOP candidates, defiantly moderate at ti..."
28146,Boyfriend Suspected in California Mom's Shooting Death,"California police suspect the teenage boyfriend of Amber Baker, a 20-year-old pregnant woman found dead in her bedroom last month, was responsible for her k..."
32762,Apple signs Oprah to make original programs,"Apple has signed a multiyear deal with Oprah Winfrey as part of the tech giant's push into original content, the company said Friday. Winfrey will create ""o..."
34966,"Poll: Among young Americans, a Clinton edge, but widespread fear for the future",A new poll offering a deep dive into the political views of the youngest American voters finds them favoring Hillary Clinton and no less likely to vote than...


Step: CNN/source-prefix removal from articles
Why this step: Remove source/location prefixes such as '(CNN)' and 'Washington (CNN)' from article text without deleting the article row.
Rows before: 54617
Rows after : 54617
Removed    : 0 (0.000% of this step input)
Cumulative removed from start: 670 (1.212%)
No rows removed in this step.


### 13.1 Clickbait-Style Headline Filter

This removes titles that are technically clean but not suitable for professional news headline generation.

In [48]:
CLICKBAIT_PATTERNS = [
    r"you won.?t believe",
    r"what happened next",
    r"this is why",
    r"everyone is talking",
    r"will shock you",
    r"the internet is going crazy",
    r"can.?t stop talking",
    r"things you need to know",
    r"here.?s why",
    r"here is why"
]

combined_clickbait_pattern = re.compile("|".join(CLICKBAIT_PATTERNS), flags=re.IGNORECASE)

before_df = model_df.copy()

model_df = model_df[
    ~model_df["title_clean"].str.contains(combined_clickbait_pattern, na=False, regex=True)
].copy()

log_filter_step(
    step_name="Clickbait-style headline filter",
    reason="Remove overly clickbait-style headlines that may weaken professional headline generation.",
    before_df=before_df,
    after_df=model_df,
    show_cols=["title_clean"]
)

Step: Clickbait-style headline filter
Why this step: Remove overly clickbait-style headlines that may weaken professional headline generation.
Rows before: 54617
Rows after : 54575
Removed    : 42 (0.077% of this step input)
Cumulative removed from start: 712 (1.288%)
Sample removed rows (up to 5):


,title_clean,article_clean_transformer,article_preview,title_preview
25,Gregg Popovich just can't stop talking about Donald Trump,"Before his team's 113-111 loss to the Golden State Warriors Sunday afternoon, San Antonio Spurs head coach Gregg Popovich popped -- ahem -- off on President...","Before his team's 113-111 loss to the Golden State Warriors Sunday afternoon, San Antonio Spurs head coach Gregg Popovich popped -- ahem -- off on President...",Gregg Popovich just can't stop talking about Donald Trump
2523,Shulkin says he was fired. Here's why it matters,Former Veterans Affairs Secretary David Shulkin has repeatedly insisted that the White House fired him from his job at the department. The White House says ...,Former Veterans Affairs Secretary David Shulkin has repeatedly insisted that the White House fired him from his job at the department. The White House says ...,Shulkin says he was fired. Here's why it matters
2972,Here's why special counsel Robert Mueller did not say whether Trump obstructed justice,Special counsel Robert Mueller's investigation did not conclude whether President Donald Trump obstructed justice after looking into numerous potential inst...,Special counsel Robert Mueller's investigation did not conclude whether President Donald Trump obstructed justice after looking into numerous potential inst...,Here's why special counsel Robert Mueller did not say whether Trump obstructed justice
4407,"Tornadoes, floods and heat: Here's why extreme weather is ravaging the US","If you feel like the weather has been out of control in much of the United States, you're right. A weather pattern that stuck around longer than usual creat...","If you feel like the weather has been out of control in much of the United States, you're right. A weather pattern that stuck around longer than usual creat...","Tornadoes, floods and heat: Here's why extreme weather is ravaging the US"
8544,Here's Why NASA Is Sending a Miniature Helicopter to Mars,Artist's concept of a Martian helicopter.Illustration: NASA JPL/CaltechNASA will be testing heavier-than-air flight on Mars by sending miniature robot helic...,Artist's concept of a Martian helicopter.Illustration: NASA JPL/CaltechNASA will be testing heavier-than-air flight on Mars by sending miniature robot helic...,Here's Why NASA Is Sending a Miniature Helicopter to Mars


## 14. Optional Language Filter (English)

All The News is mostly English, but small language noise can still appear.

- keeping rows that look like English with good confidence

In [49]:
USE_LANGUAGE_FILTER = True
# Low threshold to reduce false removals of valid English news rows
MIN_ENGLISH_PROB = 0.80

def english_probability(text):
    text = str(text).strip()
    if not text:
        return 0.0
    if not HAS_LANGDETECT:
        return np.nan
    try:
        preds = detect_langs(text)
        for p in preds:
            if str(p.lang).lower() == "en":
                return float(p.prob)
        return 0.0
    except Exception:
        return 0.0

if USE_LANGUAGE_FILTER and HAS_LANGDETECT:
    # Add english_prob to model_df first
    # Use more context (up to 1200 chars) for more stable language detection
    check_text = model_df["title_clean"] + " " + model_df["article_clean_transformer"].str[:1200]
    model_df["english_prob"] = check_text.apply(english_probability)

    # Then capture before_df *after* english_prob has been added
    before_df = model_df.copy()

    # Then apply the filter
    model_df = model_df[model_df["english_prob"] >= MIN_ENGLISH_PROB].copy()

    log_filter_step(
        step_name="English language filter",
        reason="Remove probable non-English rows to keep target language consistent for generation.",
        before_df=before_df,
        after_df=model_df,
        show_cols=["title_clean", "english_prob"]
    )
else:
    log_skipped_step(
        step_name="English language filter",
        reason="Skipped because langdetect is missing or USE_LANGUAGE_FILTER=False.",
        current_df=model_df
    )

Step: English language filter
Why this step: Remove probable non-English rows to keep target language consistent for generation.
Rows before: 54575
Rows after : 54573
Removed    : 2 (0.004% of this step input)
Cumulative removed from start: 714 (1.291%)
Sample removed rows (up to 5):


,title_clean,english_prob,article_clean_transformer,article_preview,title_preview
16450,Dólar sube a máximo de dos meses contra el yen,0.0,"Por Saqib Iqbal Ahmed NUEVA YORK, 10 jul (Reuters) - El dólar subió el lunes contra el yen, luego de que la oferta del Banco de Japón la semana pasada de co...","Por Saqib Iqbal Ahmed NUEVA YORK, 10 jul (Reuters) - El dólar subió el lunes contra el yen, luego de que la oferta del Banco de Japón la semana pasada de co...",Dólar sube a máximo de dos meses contra el yen
33455,Wall St cae luego de reporte sobre orden Trump de aplicar aranceles China,0.0,14 sep (Reuters) - Las acciones en Estados Unidos caían el viernes luego de que el presidente Donald Trump instruyó a asesores a proceder con los aranceles ...,14 sep (Reuters) - Las acciones en Estados Unidos caían el viernes luego de que el presidente Donald Trump instruyó a asesores a proceder con los aranceles ...,Wall St cae luego de reporte sobre orden Trump de aplicar aranceles China


## 15. Title-Article Alignment Check
This computes lexical overlap between title and article body.

Because abstractive headlines can have low lexical overlap, the filtering step is optional and disabled by default

In [50]:
def content_tokens(text):
    toks = re.findall(r"[A-Za-z]+", str(text).lower())
    return [t for t in toks if t not in ENGLISH_STOP_WORDS and len(t) > 2]

def title_article_overlap(title, article):
    tset = set(content_tokens(title))
    aset = set(content_tokens(article))
    if not tset:
        return 0.0
    return len(tset.intersection(aset)) / len(tset)

model_df["title_article_overlap"] = model_df.apply(
    lambda r: title_article_overlap(r["title_clean"], r["article_clean_transformer"]),
    axis=1
)

# This heuristic can remove valid abstractive headlines
# Keep it optional and using a softer threshold when enabled
USE_TITLE_ARTICLE_OVERLAP_FILTER = False
MIN_TITLE_ARTICLE_OVERLAP = 0.05

if USE_TITLE_ARTICLE_OVERLAP_FILTER:
    before_df = model_df.copy()
    model_df = model_df[model_df["title_article_overlap"] >= MIN_TITLE_ARTICLE_OVERLAP].copy()

    log_filter_step(
        step_name="Title-article overlap filter",
        reason="Optional weak-pair filter using lexical overlap (kept soft to avoid dropping abstractive headlines).",
        before_df=before_df,
        after_df=model_df,
        show_cols=["title_clean", "title_article_overlap"]
    )
else:
    log_skipped_step(
        step_name="Title-article overlap filter",
        reason="Skipped by default to preserve valid abstractive/acronym headlines.",
        current_df=model_df
    )

print("Overlap stats on current data:")
display(model_df["title_article_overlap"].describe())

Step: Title-article overlap filter
Why this step: Skipped by default to preserve valid abstractive/acronym headlines.
Step status: skipped (no rows removed).
Overlap stats on current data:


,title_article_overlap
count,54573.000000
mean,0.755041
std,0.170646
min,0.000000
25%,0.666667
50%,0.777778
75%,0.875000
max,1.000000


## 16. Optional Extra Noise Checks for Model Data

This section is still model-data preprocessing, not linguistic analysis.

It checks for remaining suspicious titles that may not be useful as headline targets.

Inspect first, then decide whether to remove more patterns.

In [51]:
remaining_noisy_title_patterns = [
    r"^\s*brief\b",
    r"^\s*update\s*\d*\s*[-:]",
    r"^\s*refile\s*-?\s*update\s*\d*\s*[-:]",
    r"^\s*wrapup\s*\d*\s*[-:]",
    r"^\s*press digest\b",
    r"^\s*factors to watch\b",
    r"^\s*preview\b",
    r"^\s*team by team analysis\b",
    r"^\s*pro rata podcast\b",
    r"^\s*cctv script\b",
    r"^\s*us stocks snapshot\b",
    r"^\s*rising:\s+[A-Za-z]+\s+\d{1,2},\s+\d{4}\s*\|\s*TheHill\s*$",
]

combined_remaining_noise_pattern = re.compile("|".join(remaining_noisy_title_patterns), flags=re.IGNORECASE)

suspicious_titles = model_df[model_df["title_clean"].str.contains(combined_remaining_noise_pattern, na=False, regex=True)]
print("Suspicious/noisy titles found:", len(suspicious_titles))
print("These are candidates for removal in the next optional step.")
display(suspicious_titles[["title_clean", "article_clean_transformer"]].head(20))

Suspicious/noisy titles found: 0
These are candidates for removal in the next optional step.


,title_clean,article_clean_transformer


In [52]:
print("Checking for 'UPDATE' and 'PREVIEW' in the flagged suspicious titles:")
mask = suspicious_titles['title_clean'].str.contains('UPDATE|PREVIEW', case=False, na=False)
display(suspicious_titles[mask][['title_clean', 'article_clean_transformer']].head(10))

Checking for 'UPDATE' and 'PREVIEW' in the flagged suspicious titles:


,title_clean,article_clean_transformer


### Remove Remaining Noisy Titles

True if the previous inspection shows real remaining noise

In [53]:
REMOVE_REMAINING_NOISY_TITLES = True

if REMOVE_REMAINING_NOISY_TITLES:
    before_df = model_df.copy()
    model_df = model_df[~model_df["title_clean"].str.contains(combined_remaining_noise_pattern, na=False, regex=True)].copy()

    log_filter_step(
        step_name="Optional noisy-title regex filter",
        reason="Remove repeated newswire-style title templates that do not behave like normal headlines.",
        before_df=before_df,
        after_df=model_df,
        show_cols=["title_clean"]
    )
else:
    log_skipped_step(
        step_name="Optional noisy-title regex filter",
        reason="Skipped because REMOVE_REMAINING_NOISY_TITLES=False.",
        current_df=model_df
    )

# Final preprocessing audit summary up to this point
audit_df = pd.DataFrame(preprocess_audit)
print("\n" + "#" * 90)
print("PREPROCESSING AUDIT SUMMARY")
print("#" * 90)
if len(audit_df) > 0:
    display(audit_df)
    final_rows_now = len(model_df)
    total_removed = INITIAL_ROWS - final_rows_now
    total_removed_pct = (total_removed / INITIAL_ROWS * 100) if INITIAL_ROWS else 0.0

    print(f"Initial rows after cleaning stage: {INITIAL_ROWS}")
    print(f"Rows remaining after all filters : {final_rows_now}")
    print(f"Total removed by filters         : {total_removed} ({total_removed_pct:.3f}%)")

    # Grand total from the raw loaded data (includes early dropna step)
    if "raw_df" in globals():
        grand_total_removed = len(raw_df) - final_rows_now
        grand_total_removed_pct = (grand_total_removed / len(raw_df) * 100) if len(raw_df) else 0.0
        print(f"Grand total removed from raw_df : {grand_total_removed} ({grand_total_removed_pct:.3f}%)")
else:
    print("No audit records found.")

Step: Optional noisy-title regex filter
Why this step: Remove repeated newswire-style title templates that do not behave like normal headlines.
Rows before: 54573
Rows after : 54573
Removed    : 0 (0.000% of this step input)
Cumulative removed from start: 714 (1.291%)
No rows removed in this step.

##########################################################################################
PREPROCESSING AUDIT SUMMARY
##########################################################################################


,step,reason,before_rows,after_rows,removed_rows,removed_pct_step,cumulative_removed,cumulative_removed_pct
0,Safety char-length filter,Remove rows that become too short after cleaning; very short body/title pairs are usually weak supervision.,55287,55287,0,0.000,0,0.000
1,Exact duplicate removal,Remove repeated article pairs to reduce memorization and data leakage across splits.,55287,55166,121,0.219,121,0.219
2,Loose near-duplicate removal,Catch practical near-duplicates with tiny edits so the model sees more diverse examples.,55166,55041,125,0.227,246,0.445
3,Embedding-based near-duplicate removal,Remove semantically similar duplicate stories that text-based dedup may miss.,55041,54902,139,0.253,385,0.696
4,Word-length filter,Keep rows where article has enough context and headline length looks realistic for training.,54902,54874,28,0.051,413,0.747
5,"Tokenizer policy (truncate long, drop extreme outliers)",Word count is approximate; token counts match transformer behavior. We keep more data by truncating normal long inputs and dropping only extreme outliers.,54874,54618,256,0.467,669,1.210
6,Headline quality filter,"Remove headline targets that look noisy or low-information (generic labels, punctuation-heavy, etc.).",54618,54617,1,0.002,670,1.212
7,CNN/source-prefix removal from articles,Remove source/location prefixes such as '(CNN)' and 'Washington (CNN)' from article text without deleting the article row.,54617,54617,0,0.000,670,1.212
8,Clickbait-style headline filter,Remove overly clickbait-style headlines that may weaken professional headline generation.,54617,54575,42,0.077,712,1.288
9,English language filter,Remove probable non-English rows to keep target language consistent for generation.,54575,54573,2,0.004,714,1.291


Initial rows after cleaning stage: 55287
Rows remaining after all filters : 54573
Total removed by filters         : 714 (1.291%)
Grand total removed from raw_df : 714 (1.291%)


### 16.1 Publication Balance Analysis

This checks if one publisher dominates the dataset.
A dominated dataset may make the model learn one publisher's headline style too strongly.

In [54]:
if "publication" in model_df.columns:
    print("Publication distribution:")
    publication_counts = model_df["publication"].fillna("Unknown").value_counts()
    display(publication_counts.head(20))

    print("Publication percentage:")
    display((publication_counts / len(model_df) * 100).round(2).head(20))
else:
    print("No publication column found.")

Publication distribution:


,count
publication,
Reuters,21243
CNN,9282
CNBC,7470
People,6869
The New York Times,3398
The Verge,2731
Fox News,1678
Gizmodo,945
Wired,734


Publication percentage:


,count
publication,
Reuters,38.93
CNN,17.01
CNBC,13.69
People,12.59
The New York Times,6.23
The Verge,5.00
Fox News,3.07
Gizmodo,1.73
Wired,1.34


### 16.2 Section / Category Balance Analysis

This checks whether the dataset is dominated by one topic.

In [55]:
if "section" in model_df.columns:
    print("Section distribution:")
    section_counts = model_df["section"].fillna("Unknown").value_counts()
    display(section_counts.head(30))

    print("Section percentage:")
    display((section_counts / len(model_df) * 100).round(2).head(30))
else:
    print("No section column found.")

Section distribution:


,count
section,
World News,7773
politics,7387
crime,5996
Tech,5488
Politics,5203
Westlaw News,3130
Technology News,3112
U.S. Legal News,2973
us,2422


Section percentage:


,count
section,
World News,14.24
politics,13.54
crime,10.99
Tech,10.06
Politics,9.53
Westlaw News,5.74
Technology News,5.70
U.S. Legal News,5.45
us,4.44


## 17. Create Model-Specific Input/Target Columns (All Models)

Instead of selecting one model, this step creates model-ready columns for all configured models in one dataframe.

For each model alias, we create:

- `{alias}_article`
- `{alias}_title`

Formatting rules:

- seq2seq models (T5/FLAN/BART): `summarize: ...`
- causal models (GPT-2): `Article: ...\nHeadline:`

Optional model-specific token truncation is applied so each model column respects its own input limit.

In [56]:
# Build model-ready columns for all configured model families
# We keep one cleaned base article/title, then format them per model
MODEL_CONFIGS = {
    "flan_t5_small": {
        "checkpoint": "google/flan-t5-small",
        "family": "seq2seq",
        "prefix": "summarize: "
    },
    "flan_t5_base": {
        "checkpoint": "google/flan-t5-base",
        "family": "seq2seq",
        "prefix": "summarize: "
    },
    "t5_small": {
        "checkpoint": "t5-small",
        "family": "seq2seq",
        "prefix": "summarize: "
    },
    "t5_base": {
        "checkpoint": "t5-base",
        "family": "seq2seq",
        "prefix": "summarize: "
    },
    "bart_base": {
        "checkpoint": "facebook/bart-base",
        "family": "seq2seq",
        "prefix": "summarize: "
    },
    "gpt2": {
        "checkpoint": "gpt2",
        "family": "causal",
        "prefix": None
    },
}

# If this exists, it means long texts were already truncated upstream
base_article_series = model_df["article_clean_transformer"].astype(str)
base_title_series = model_df["title_clean"].astype(str)

# Optional per-model truncation for each article column
APPLY_MODEL_SPECIFIC_TRUNCATION = True

def truncate_text_by_tokens(text, tokenizer, max_tokens):
    ids = tokenizer(
        str(text),
        add_special_tokens=True,
        truncation=True,
        max_length=max_tokens
    )["input_ids"]
    return tokenizer.decode(ids, skip_special_tokens=True, clean_up_tokenization_spaces=True)

created_article_cols = []
created_title_cols = []
tokenizer_cache = {}

for alias, cfg_model in MODEL_CONFIGS.items():
    checkpoint = cfg_model["checkpoint"]
    family = cfg_model["family"]

    limits = MODEL_TOKEN_LIMITS.get(checkpoint, {})
    max_input_tokens = limits.get("max_input", DEFAULT_MAX_INPUT_TOKENS)

    article_series = base_article_series.copy()

    if APPLY_MODEL_SPECIFIC_TRUNCATION and HAS_TRANSFORMERS:
        if checkpoint not in tokenizer_cache:
            tokenizer_cache[checkpoint] = AutoTokenizer.from_pretrained(checkpoint, use_fast=True)
        tok = tokenizer_cache[checkpoint]
        article_series = article_series.apply(lambda x: truncate_text_by_tokens(x, tok, max_input_tokens))

    article_col = f"{alias}_article"
    title_col = f"{alias}_title"

    if family == "seq2seq":
        model_df[article_col] = cfg_model["prefix"] + article_series
    elif family == "causal":
        model_df[article_col] = "Article: " + article_series + "\nHeadline:"
    else:
        raise ValueError(f"Unsupported family: {family}")

    model_df[title_col] = base_title_series

    created_article_cols.append(article_col)
    created_title_cols.append(title_col)

paired_model_columns = [c for pair in zip(created_article_cols, created_title_cols) for c in pair]

print("Final model dataframe shape:", model_df.shape)
print("Created model-specific pair columns:")
print(paired_model_columns)
display(model_df[paired_model_columns[:4]].head(3))

Final model dataframe shape: (54573, 41)
Created model-specific pair columns:
['flan_t5_small_article', 'flan_t5_small_title', 'flan_t5_base_article', 'flan_t5_base_title', 't5_small_article', 't5_small_title', 't5_base_article', 't5_base_title', 'bart_base_article', 'bart_base_title', 'gpt2_article', 'gpt2_title']


,flan_t5_small_article,flan_t5_small_title,flan_t5_base_article,flan_t5_base_title
0,summarize: A unit of Chinese ride-hailing firm Didi Chuxing has submitted an application to raise 10 billion yuan ($1.6 billion) through an issuance of asse...,China's Didi looks to raise $1.6 billion via asset-backed securities,summarize: A unit of Chinese ride-hailing firm Didi Chuxing has submitted an application to raise 10 billion yuan ($1.6 billion) through an issuance of asse...,China's Didi looks to raise $1.6 billion via asset-backed securities
1,"summarize: In a new interview with Vanity Fair, President Trump's short-lived White House Communications Director Anthony Scaramucci recalls his 10 tumultuo...",Anthony Scaramucci Dishes on His Firing in Vanity Fair Interview,"summarize: In a new interview with Vanity Fair, President Trump's short-lived White House Communications Director Anthony Scaramucci recalls his 10 tumultuo...",Anthony Scaramucci Dishes on His Firing in Vanity Fair Interview
2,"summarize: Rep. Duncan Hunter, R-Ca, is expected to appear in federal court on Tuesday to plead guilty for spending more than $200,000 in campaign funds on ...",Duncan Hunter expected to change plea to guilty in federal court,"summarize: Rep. Duncan Hunter, R-Ca, is expected to appear in federal court on Tuesday to plead guilty for spending more than $200,000 in campaign funds on ...",Duncan Hunter expected to change plea to guilty in federal court


## 18. Final Sanity Checks

These checks make sure the final model columns are clean and usable.

In [57]:
model_pair_columns = [
    c for c in model_df.columns
    if any(c.endswith(suffix) for suffix in ["_article", "_title"])
]

article_cols = [c for c in model_pair_columns if c.endswith("_article")]
title_cols = [c for c in model_pair_columns if c.endswith("_title")]

for col in article_cols:
    print(f"Missing {col}:", model_df[col].isna().sum())
    print(f"Empty {col}:", (model_df[col].str.strip() == "").sum())

for col in title_cols:
    print(f"Missing {col}:", model_df[col].isna().sum())
    print(f"Empty {col}:", (model_df[col].str.strip() == "").sum())

for article_col in article_cols:
    base = article_col.replace("_article", "")
    title_col = f"{base}_title"
    if title_col in model_df.columns:
        dup_pairs = model_df[[article_col, title_col]].duplicated().sum()
        print(f"Duplicate pairs for {base}:", dup_pairs)

print("Final shape:", model_df.shape)

Missing flan_t5_small_article: 0
Empty flan_t5_small_article: 0
Missing flan_t5_base_article: 0
Empty flan_t5_base_article: 0
Missing t5_small_article: 0
Empty t5_small_article: 0
Missing t5_base_article: 0
Empty t5_base_article: 0
Missing bart_base_article: 0
Empty bart_base_article: 0
Missing gpt2_article: 0
Empty gpt2_article: 0
Missing flan_t5_small_title: 0
Empty flan_t5_small_title: 0
Missing flan_t5_base_title: 0
Empty flan_t5_base_title: 0
Missing t5_small_title: 0
Empty t5_small_title: 0
Missing t5_base_title: 0
Empty t5_base_title: 0
Missing bart_base_title: 0
Empty bart_base_title: 0
Missing gpt2_title: 0
Empty gpt2_title: 0
Duplicate pairs for flan_t5_small: 0
Duplicate pairs for flan_t5_base: 0
Duplicate pairs for t5_small: 0
Duplicate pairs for t5_base: 0
Duplicate pairs for bart_base: 0
Duplicate pairs for gpt2: 0
Final shape: (54573, 41)


## 19. Keep Only Useful Columns for the Model Files

 we keep only model-specific article/title pairs

So the exported files contain only columns like:

- `flan_t5_base_article`, `flan_t5_base_title`
- `bart_base_article`, `bart_base_title`
- `gpt2_article`, `gpt2_title`

No metadata or analysis columns are kept in final model files

In [58]:
cols_to_save = [
    c for c in model_df.columns
    if any(c.endswith(suffix) for suffix in ["_article", "_title"])
]

model_ready_df = model_df[cols_to_save].copy()

print("Columns saved for model-ready data (article/title pairs only):")
print(model_ready_df.columns.tolist())
print("Shape:", model_ready_df.shape)
display(model_ready_df.head(3))

Columns saved for model-ready data (article/title pairs only):
['flan_t5_small_article', 'flan_t5_small_title', 'flan_t5_base_article', 'flan_t5_base_title', 't5_small_article', 't5_small_title', 't5_base_article', 't5_base_title', 'bart_base_article', 'bart_base_title', 'gpt2_article', 'gpt2_title']
Shape: (54573, 12)


,flan_t5_small_article,flan_t5_small_title,flan_t5_base_article,flan_t5_base_title,t5_small_article,t5_small_title,t5_base_article,t5_base_title,bart_base_article,bart_base_title,gpt2_article,gpt2_title
0,summarize: A unit of Chinese ride-hailing firm Didi Chuxing has submitted an application to raise 10 billion yuan ($1.6 billion) through an issuance of asse...,China's Didi looks to raise $1.6 billion via asset-backed securities,summarize: A unit of Chinese ride-hailing firm Didi Chuxing has submitted an application to raise 10 billion yuan ($1.6 billion) through an issuance of asse...,China's Didi looks to raise $1.6 billion via asset-backed securities,summarize: A unit of Chinese ride-hailing firm Didi Chuxing has submitted an application to raise 10 billion yuan ($1.6 billion) through an issuance of asse...,China's Didi looks to raise $1.6 billion via asset-backed securities,summarize: A unit of Chinese ride-hailing firm Didi Chuxing has submitted an application to raise 10 billion yuan ($1.6 billion) through an issuance of asse...,China's Didi looks to raise $1.6 billion via asset-backed securities,summarize: A unit of Chinese ride-hailing firm Didi Chuxing has submitted an application to raise 10 billion yuan ($1.6 billion) through an issuance of asse...,China's Didi looks to raise $1.6 billion via asset-backed securities,Article: A unit of Chinese ride-hailing firm Didi Chuxing has submitted an application to raise 10 billion yuan ($1.6 billion) through an issuance of asset-...,China's Didi looks to raise $1.6 billion via asset-backed securities
1,"summarize: In a new interview with Vanity Fair, President Trump's short-lived White House Communications Director Anthony Scaramucci recalls his 10 tumultuo...",Anthony Scaramucci Dishes on His Firing in Vanity Fair Interview,"summarize: In a new interview with Vanity Fair, President Trump's short-lived White House Communications Director Anthony Scaramucci recalls his 10 tumultuo...",Anthony Scaramucci Dishes on His Firing in Vanity Fair Interview,"summarize: In a new interview with Vanity Fair, President Trump's short-lived White House Communications Director Anthony Scaramucci recalls his 10 tumultuo...",Anthony Scaramucci Dishes on His Firing in Vanity Fair Interview,"summarize: In a new interview with Vanity Fair, President Trump's short-lived White House Communications Director Anthony Scaramucci recalls his 10 tumultuo...",Anthony Scaramucci Dishes on His Firing in Vanity Fair Interview,"summarize: In a new interview with Vanity Fair, President Trump's short-lived White House Communications Director Anthony Scaramucci recalls his 10 tumultuo...",Anthony Scaramucci Dishes on His Firing in Vanity Fair Interview,"Article: In a new interview with Vanity Fair, President Trump's short-lived White House Communications Director Anthony Scaramucci recalls his 10 tumultuous...",Anthony Scaramucci Dishes on His Firing in Vanity Fair Interview
2,"summarize: Rep. Duncan Hunter, R-Ca, is expected to appear in federal court on Tuesday to plead guilty for spending more than $200,000 in campaign funds on ...",Duncan Hunter expected to change plea to guilty in federal court,"summarize: Rep. Duncan Hunter, R-Ca, is expected to appear in federal court on Tuesday to plead guilty for spending more than $200,000 in campaign funds on ...",Duncan Hunter expected to change plea to guilty in federal court,"summarize: Rep. Duncan Hunter, R-Ca, is expected to appear in federal court on Tuesday to plead guilty for spending more than $200,000 in campaign funds on ...",Duncan Hunter expected to change plea to guilty in federal court,"summarize: Rep. Duncan Hunter, R-Ca, is expected to appear in federal court on Tuesday to plead guilty for spending more than $200,000 in campaign funds on ...",Duncan Hunter expected to change plea to guilty in federal court,"summarize: Rep. Duncan Hunter, R-Ca, is expected to appear in federal court on Tuesday to plead guilty for spending more than $200,000 in campaign

## 20. Train / Validation / Test Split

 split:

- 80% training
- 10% validation
- 10% testing

Split strategy is configurable:

- `temporal` (default): oldest news in train, newest in validation/test (more realistic)
- `group_random`: group-aware random split using `dedup_key_loose`
- `random`: standard random split fallback

## 21. Model-Ready Handoff Checks

This check makes sure the final dataset is immediately usable for training

I verify that:

- only model pair columns exist (`*_article`, `*_title`)
- every model alias has both columns
- there are no missing/empty values in train/validation/test

In [59]:
def validate_model_ready_split(df, split_name):
    pair_cols = [c for c in df.columns if c.endswith("_article") or c.endswith("_title")]
    non_pair_cols = [c for c in df.columns if c not in pair_cols]

    if non_pair_cols:
        raise ValueError(f"{split_name}: found non-pair columns: {non_pair_cols}")

    aliases = sorted({c.rsplit("_", 1)[0] for c in pair_cols})
    missing_pairs = []
    for a in aliases:
        if f"{a}_article" not in df.columns or f"{a}_title" not in df.columns:
            missing_pairs.append(a)

    if missing_pairs:
        raise ValueError(f"{split_name}: missing pair columns for aliases: {missing_pairs}")

    for a in aliases:
        ac = f"{a}_article"
        tc = f"{a}_title"
        na_count = int(df[[ac, tc]].isna().any(axis=1).sum())
        empty_count = int(((df[ac].astype(str).str.strip() == "") | (df[tc].astype(str).str.strip() == "")).sum())
        if na_count > 0 or empty_count > 0:
            raise ValueError(
                f"{split_name}: alias {a} has {na_count} NA rows and {empty_count} empty rows"
            )

    print(f"{split_name}: ready ({len(aliases)} model aliases, {len(pair_cols)} columns, {len(df)} rows)")


SPLIT_STRATEGY = "group_random"  # options: "temporal", "group_random", "random"

if SPLIT_STRATEGY == "temporal":
    if "date" not in model_df.columns:
        raise ValueError("Temporal split requested but `date` column is missing in model_df.")

    split_dates = pd.to_datetime(model_df["date"], errors="coerce")
    valid_mask = split_dates.notna()

    if not valid_mask.all():
        print(f"Temporal split warning: {(~valid_mask).sum()} rows have invalid dates and will be placed at the end.")

    sort_keys = split_dates.fillna(pd.Timestamp.max)
    ordered_pos = np.argsort(sort_keys.values)

    n = len(ordered_pos)
    train_end = int(n * 0.80)
    val_end = int(n * 0.90)

    train_pos = ordered_pos[:train_end]
    val_pos = ordered_pos[train_end:val_end]
    test_pos = ordered_pos[val_end:]

    train_df = model_ready_df.iloc[train_pos].copy()
    val_df = model_ready_df.iloc[val_pos].copy()
    test_df = model_ready_df.iloc[test_pos].copy()

    print("Used temporal split (oldest->train, newest->validation/test).")

elif SPLIT_STRATEGY == "group_random" and "dedup_key_loose" in model_df.columns:
    # Use grouping from model_df to avoid leakage,
    # while train/val/test files keep only model pair columns.
    gss_1 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
    train_pos, temp_pos = next(gss_1.split(model_df, groups=model_df["dedup_key_loose"]))

    train_df = model_ready_df.iloc[train_pos].copy()
    temp_ready_df = model_ready_df.iloc[temp_pos].copy()
    temp_groups = model_df.iloc[temp_pos]["dedup_key_loose"].reset_index(drop=True)

    gss_2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
    val_rel_pos, test_rel_pos = next(gss_2.split(temp_ready_df, groups=temp_groups))

    val_df = temp_ready_df.iloc[val_rel_pos].copy()
    test_df = temp_ready_df.iloc[test_rel_pos].copy()

    print("Used group-aware random split with dedup_key_loose.")

else:
    train_df, temp_df = train_test_split(
        model_ready_df,
        test_size=0.20,
        random_state=42,
        shuffle=True
    )

    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        random_state=42,
        shuffle=True
    )

    print("Used normal random split.")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("Split percentages:")
print("Train %:", round(len(train_df) / len(model_ready_df) * 100, 2))
print("Validation %:", round(len(val_df) / len(model_ready_df) * 100, 2))
print("Test %:", round(len(test_df) / len(model_ready_df) * 100, 2))

validate_model_ready_split(model_ready_df, "full")
validate_model_ready_split(train_df, "train")
validate_model_ready_split(val_df, "validation")
validate_model_ready_split(test_df, "test")

Used group-aware random split with dedup_key_loose.
Train: (43658, 12)
Validation: (5457, 12)
Test: (5458, 12)
Split percentages:
Train %: 80.0
Validation %: 10.0
Test %: 10.0
full: ready (6 model aliases, 12 columns, 54573 rows)
train: ready (6 model aliases, 12 columns, 43658 rows)
validation: ready (6 model aliases, 12 columns, 5457 rows)
test: ready (6 model aliases, 12 columns, 5458 rows)


In [60]:
SPLIT_STRATEGY = "group_random"  # options: "temporal", "group_random", "random"

if SPLIT_STRATEGY == "temporal":
    if "date" not in model_df.columns:
        raise ValueError("Temporal split requested but `date` column is missing in model_df.")

    split_dates = pd.to_datetime(model_df["date"], errors="coerce")
    valid_mask = split_dates.notna()

    if not valid_mask.all():
        print(f"Temporal split warning: {(~valid_mask).sum()} rows have invalid dates and will be placed at the end.")

    sort_keys = split_dates.fillna(pd.Timestamp.max)
    ordered_pos = np.argsort(sort_keys.values)

    n = len(ordered_pos)
    train_end = int(n * 0.80)
    val_end = int(n * 0.90)

    train_pos = ordered_pos[:train_end]
    val_pos = ordered_pos[train_end:val_end]
    test_pos = ordered_pos[val_end:]

    train_df = model_ready_df.iloc[train_pos].copy()
    val_df = model_ready_df.iloc[val_pos].copy()
    test_df = model_ready_df.iloc[test_pos].copy()

    print("Used temporal split (oldest->train, newest->validation/test).")

elif SPLIT_STRATEGY == "group_random" and "dedup_key_loose" in model_df.columns:
    # Use grouping from model_df to avoid leakage,
    # while train/val/test files keep only model pair columns.
    gss_1 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
    train_pos, temp_pos = next(gss_1.split(model_df, groups=model_df["dedup_key_loose"]))

    train_df = model_ready_df.iloc[train_pos].copy()
    temp_ready_df = model_ready_df.iloc[temp_pos].copy()
    temp_groups = model_df.iloc[temp_pos]["dedup_key_loose"].reset_index(drop=True)

    gss_2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
    val_rel_pos, test_rel_pos = next(gss_2.split(temp_ready_df, groups=temp_groups))

    val_df = temp_ready_df.iloc[val_rel_pos].copy()
    test_df = temp_ready_df.iloc[test_rel_pos].copy()

    print("Used group-aware random split with dedup_key_loose.")

else:
    train_df, temp_df = train_test_split(
        model_ready_df,
        test_size=0.20,
        random_state=42,
        shuffle=True
    )

    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        random_state=42,
        shuffle=True
    )

    print("Used normal random split.")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("Split percentages:")
print("Train %:", round(len(train_df) / len(model_ready_df) * 100, 2))
print("Validation %:", round(len(val_df) / len(model_ready_df) * 100, 2))
print("Test %:", round(len(test_df) / len(model_ready_df) * 100, 2))

Used group-aware random split with dedup_key_loose.
Train: (43658, 12)
Validation: (5457, 12)
Test: (5458, 12)
Split percentages:
Train %: 80.0
Validation %: 10.0
Test %: 10.0


## 22. Save Final Model Data Files

These files are ready for direct model training.

two formats are saved:

- wide files with all model pair columns (`*_article`, `*_title`)
- per-model files with standardized columns (`model_input`, `model_target`)

In [61]:
train_path = os.path.join(OUTPUT_DIR, "news_headline_train_wide.csv")
val_path = os.path.join(OUTPUT_DIR, "news_headline_validation_wide.csv")
test_path = os.path.join(OUTPUT_DIR, "news_headline_test_wide.csv")
full_path = os.path.join(OUTPUT_DIR, "news_headline_model_ready_full_wide.pkl")

# 1) Save wide files (all model pairs in one table)
train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)
model_ready_df.to_pickle(full_path)

# 2) Save per-model files with standardized names for immediate training scripts
#    Each file has exactly two columns: model_input, model_target
aliases = sorted({c.rsplit("_", 1)[0] for c in model_ready_df.columns if c.endswith("_article")})
for alias in aliases:
    article_col = f"{alias}_article"
    title_col = f"{alias}_title"

    if article_col not in model_ready_df.columns or title_col not in model_ready_df.columns:
        continue

    alias_train = train_df[[article_col, title_col]].rename(columns={article_col: "model_input", title_col: "model_target"})
    alias_val = val_df[[article_col, title_col]].rename(columns={article_col: "model_input", title_col: "model_target"})
    alias_test = test_df[[article_col, title_col]].rename(columns={article_col: "model_input", title_col: "model_target"})

    alias_train.to_csv(os.path.join(OUTPUT_DIR, f"{alias}_train.csv"), index=False)
    alias_val.to_csv(os.path.join(OUTPUT_DIR, f"{alias}_validation.csv"), index=False)
    alias_test.to_csv(os.path.join(OUTPUT_DIR, f"{alias}_test.csv"), index=False)

print("Saved model-ready files:")
print(train_path)
print(val_path)
print(test_path)
print(full_path)
print("Per-model files saved for aliases:", aliases)

Saved model-ready files:
/content/drive/MyDrive/headlineGenerationProjectNLP/news_headline_model_data/news_headline_train_wide.csv
/content/drive/MyDrive/headlineGenerationProjectNLP/news_headline_model_data/news_headline_validation_wide.csv
/content/drive/MyDrive/headlineGenerationProjectNLP/news_headline_model_data/news_headline_test_wide.csv
/content/drive/MyDrive/headlineGenerationProjectNLP/news_headline_model_data/news_headline_model_ready_full_wide.pkl
Per-model files saved for aliases: ['bart_base', 'flan_t5_base', 'flan_t5_small', 'gpt2', 't5_base', 't5_small']
